In [0]:
import dlt
from pyspark.sql.functions import col, current_timestamp

# Source views for each machine type
@dlt.view
def cnc_sensor_source():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/Volumes/workspace/default/iot_sensor_data/*/*/*/*.json")
        .filter(col("machine_type") == "CNC")
        .withColumn("Last_timestamp", current_timestamp())
    )

@dlt.view
def transformer_sensor_source():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/Volumes/workspace/default/iot_sensor_data/*/*/*/*.json")
        .filter(col("machine_type") == "Transformer")
        .withColumn("Last_timestamp", current_timestamp())
    )

@dlt.view
def motor_sensor_source():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("/Volumes/workspace/default/iot_sensor_data/*/*/*/*.json")
        .filter(col("machine_type") == "Motor")
        .withColumn("Last_timestamp", current_timestamp())
    )

# Create streaming tables for each machine type
dlt.create_streaming_table("cnc_sensor_cdc")
dlt.create_streaming_table("transformer_sensor_cdc")
dlt.create_streaming_table("motor_sensor_cdc")

# Auto CDC flows with SCD Type 2 for each table
dlt.create_auto_cdc_flow(
    target="cnc_sensor_cdc",
    source="cnc_sensor_source",
    keys=["sensor_event_id"],
    sequence_by=col("publish_timestamp"),
    stored_as_scd_type=2
)

dlt.create_auto_cdc_flow(
    target="transformer_sensor_cdc",
    source="transformer_sensor_source",
    keys=["sensor_event_id"],
    sequence_by=col("publish_timestamp"),
    stored_as_scd_type=2
)

dlt.create_auto_cdc_flow(
    target="motor_sensor_cdc",
    source="motor_sensor_source",
    keys=["sensor_event_id"],
    sequence_by=col("publish_timestamp"),
    stored_as_scd_type=2
)